# Franka：建模、符号导出与参数分解

使用无夹爪的 `panda_arm.xml`，验证正运动学、雅可比和刚体动力学，导出 Python/C/LaTeX，再用生成代码做理想/加噪数据辨识与 `M/c/g` 分解。模型有 70 个刚体参数、43 个基参数；末帧对应 `link7`（MDH 末项 `d=0`）。

- 输入：仓库内的 MDH 参数与 MJCF；可选恢复读取 `configs/franka/recovery.json`。
- 输出：`outputs/franka_symbolic/` 下的导出代码，以及其 `identification/` 子目录中的轨迹、训练/验证数据和报告。第一个代码单元创建输出目录；再次运行写文件单元会覆盖同名文件，可先修改 `OUT`。
- 顺序：第 1–8 节依次运行；第 3–4 节的符号生成与导出耗时较长，第 6 节的验证轨迹搜索也包含 SLSQP。第 9 节恢复默认跳过，按需开启。

依赖：`pip install -e ".[sim,ident,viz,dev]"`。只想快速辨识可运行 [identification.ipynb](identification.ipynb)；同类 3 轴示例见 [RRR 建模](../rrr/modeling.ipynb)。README 的 62 基参数闭环结果由 `robot-identify --robot franka` 生成。


In [2]:
import sys
import time
from pathlib import Path

import numpy as np
import sympy

# 根据项目标记定位根目录，支持从仓库或 notebook 目录启动。
def _project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "robot_model" / "__init__.py"
        ).is_file():
            return candidate
    raise FileNotFoundError("找不到含 pyproject.toml 和 src/robot_model 的项目根目录")


_ROOT = _project_root(Path.cwd().resolve())
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import mujoco

from robot_model import Robot
from robot_model.robots.franka import (
    PANDA_MDH_BODY_NAMES,
    PANDA_MDH_PARMS,
    panda_xml_path,
)
from robot_model.simulation.reference import MujocoReference, strip_to_rigid_body

sympy.init_printing()
OUT = _ROOT / "outputs" / "franka_symbolic"
OUT.mkdir(parents=True, exist_ok=True)
print("robot_model:", _ROOT)
print("导出目录:", OUT)

robot_model: /home/xiaomeng/code/robot_model
导出目录: /home/xiaomeng/code/robot_model/outputs/franka_symbolic


## 1. 正运动学

MDH 参数来自 `src/robot_model/robots/franka.py`，逐帧与 MuJoCo 的 `link1..link7` 原点比较。


In [3]:
robot = Robot.from_mdh("franka_panda_arm", PANDA_MDH_PARMS)
robot.print_dh_table()
print("dof =", robot.dof, " convention =", robot.convention)
print("关节变量:", list(robot.q))

Joint,a,d,α,θ,关节类型
Joint1,0,0.333,0,q1,转动关节
Joint2,0,0,-pi/2,q2,转动关节
Joint3,0,0.316,pi/2,q3,转动关节
Joint4,0.0825,0,pi/2,q4,转动关节
Joint5,-0.0825,0.384,-pi/2,q5,转动关节
Joint6,0,0,pi/2,q6,转动关节
Joint7,0.088,0,pi/2,q7,转动关节


dof = 7  convention = modified
关节变量: [q1, q2, q3, q4, q5, q6, q7]


In [ ]:
T_end = robot.forward_kinematics()
print("T(q) 节点数", sum(1 for _ in sympy.preorder_traversal(T_end)))
print("末端位置各分量字符数", [len(str(T_end[i, 3])) for i in range(3)])

# 展示末端高度表达式，其余位姿元素保留在 T_end。
sympy.trigsimp(T_end[2, 3])


MJCF 带有电机惯量、阻尼、碰撞和限位约束。本页使用 `strip_to_rigid_body` 去除这些项，使 `mj_inverse` 对应刚体模型的 `M ddq + c + g`，便于逐项核对。


In [5]:
model = strip_to_rigid_body(mujoco.MjModel.from_xml_path(str(panda_xml_path())))
ref = MujocoReference(model, PANDA_MDH_BODY_NAMES)

q_test = np.array([0.3, -0.6, 0.4, -1.2, 0.5, 1.0, -0.4])
T_mj = ref.body_poses(q_test)
for i in range(robot.dof):
    T_sym = robot.fk_numpy(q_test, link=i)
    pos_err = np.linalg.norm(T_sym[:3, 3] - T_mj[i][:3, 3])
    rot_err = np.linalg.norm(T_sym[:3, :3] - T_mj[i][:3, :3])
    print(f"link{i + 1}: 位置误差 {pos_err:.2e}  姿态误差 {rot_err:.2e}")

link1: 位置误差 0.00e+00  姿态误差 1.92e-16
link2: 位置误差 0.00e+00  姿态误差 9.56e-16
link3: 位置误差 1.69e-16  姿态误差 1.07e-15
link4: 位置误差 1.44e-16  姿态误差 1.50e-15
link5: 位置误差 4.53e-16  姿态误差 1.87e-15
link6: 位置误差 4.53e-16  姿态误差 2.10e-15
link7: 位置误差 6.81e-16  姿态误差 2.42e-15


In [6]:
# 顺带验证 MDH → PoE 互转后正运动学不变
poe = robot.to_poe()
cmp = robot.compare_fk(poe, q_test)
print(f"MDH vs PoE: 位置 {cmp['pos_err']:.2e}  姿态 {cmp['rot_err']:.2e}")

MDH vs PoE: 位置 2.63e-16  姿态 0.00e+00


## 2. 雅可比

几何雅可比满足 `[v; ω] = J dq`；解析雅可比满足 `[ṗ; φ̇] = Ja dq`，其中 `φ` 为 ZYX 的 `[roll, pitch, yaw]`。两者通过 `ω = E(φ) φ̇` 换算，非奇异姿态下 `Ja = [Jp; E(φ)⁻¹ Jo]`。

先将几何雅可比与 `mj_jacBody` 比较，再把姿态有限差分得到的 `ja_numpy` 与几何换算结果比较。后者含差分误差，应在数值容差内一致。


In [9]:
kin = robot.kinematics
J_end = kin.J[-1]
print("J 形状", J_end.shape, " 节点数", sum(1 for _ in sympy.preorder_traversal(J_end)))

J_fun = sympy.lambdify(list(robot.q), J_end, "numpy")
J_sym = np.asarray(J_fun(*q_test), dtype=float)

ref.forward(q_test)
bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "link7")
jacp = np.zeros((3, model.nv))
jacr = np.zeros((3, model.nv))
mujoco.mj_jacBody(model, ref.data, jacp, jacr, bid)
J_mj = np.vstack([jacp, jacr])

print(f"几何 J vs mj_jacBody 误差 {np.abs(J_sym - J_mj).max():.2e}")

dq_test = np.array([0.4, -0.3, 0.5, 0.2, -0.6, 0.3, 0.1])
twist = J_sym @ dq_test  # [v; omega]
v, omega = twist[:3], twist[3:]
print("末端 twist [v; omega]:", np.round(twist, 4))

# 解析与几何：ω = E(φ) φ̇  ⇔  φ̇ = E(φ)^{-1} ω
# E 依赖当前姿态 φ，不是常数矩阵，也不是转置。
# 解析雅可比由几何雅可比换算：Ja = [Jp; E^{-1} Jo]
from robot_model.utils.mathutil import rpy_zyx_E

Ja_fd = kin.ja_numpy(q_test)
Ja_from_J = kin.ja_from_geometric_numpy(q_test)
xa_dot = Ja_fd @ dq_test  # [ṗ; φ̇]
phidot = xa_dot[3:]

phi = np.asarray(
    sympy.lambdify(list(robot.q), kin.rpy[-1], "numpy")(*q_test), dtype=float
).reshape(3)
r, p, y = sympy.symbols("roll pitch yaw", real=True)
E = np.asarray(
    sympy.lambdify((r, p, y), rpy_zyx_E([r, p, y]), "numpy")(*phi), dtype=float
)
Einv = np.linalg.inv(E)

print(f"解析 Ja：差分 vs [Jp; E^{{-1}} Jo]     误差 {np.abs(Ja_fd - Ja_from_J).max():.2e}")
print(f"直接比 φ̇ 与 ω（本来就不同）         误差 {np.abs(phidot - omega).max():.2e}")
print(f"E^{{-1}} @ ω  → 应等于 φ̇            误差 {np.abs(Einv @ omega - phidot).max():.2e}")
print(f"E @ φ̇     → 应等于 ω             误差 {np.abs(E @ phidot - omega).max():.2e}")
print("末端 [ṗ; φ̇]:", np.round(xa_dot, 4))
print("φ (rpy):", np.round(phi, 4))
print("E(φ):\n", np.round(E, 4))

J 形状 (6, 7)  节点数 3978
几何 J vs mj_jacBody 误差 2.66e-15
末端 twist [v; omega]: [-0.4836  0.0692  0.0314 -0.0231 -0.9453  0.2666]
解析 Ja：差分 vs [Jp; E^{-1} Jo]     误差 3.63e-08
直接比 φ̇ 与 ω（本来就不同）         误差 9.73e-01
E^{-1} @ ω  → 应等于 φ̇            误差 1.45e-08
E @ φ̇     → 应等于 ω             误差 1.54e-08
末端 [ṗ; φ̇]: [-0.4836  0.0692  0.0314 -0.9963 -0.1124  0.6   ]
φ (rpy): [ 3.0899 -0.3412  1.4271]
E(φ):
 [[ 0.1349 -0.9897  0.    ]
 [ 0.9326  0.1432  0.    ]
 [ 0.3347  0.      1.    ]]


## 3. 符号动力学

慢速单元：生成 `M/c/g/tau/C/H` 并统计表达式规模。7 轴表达式较大，这里保留未化简形式，避免额外的全局 `simplify` 或 `trigsimp`。


In [ ]:
dyn = robot.dynamics
for key, gen_fn in [
    ("M", dyn.gen_inertiamatrix),
    ("g", dyn.gen_gravityterm),
    ("c", dyn.gen_coriolisterm),
    ("tau", dyn.gen_invdyn),
    ("C", dyn.gen_coriolismatrix),
    ("H", dyn.gen_regressor),
]:
    t0 = time.time()
    expr = gen_fn()
    n_ops = sum(1 for _ in sympy.preorder_traversal(expr))
    print(f"{key:4s} {tuple(expr.shape)!s:8s} {time.time() - t0:6.1f}s  {n_ops:>9,d} 个节点", flush=True)

print("\n动力学参数个数:", dyn.n_dynparms, " 前 10 个:", list(dyn.dynparms)[:10])

In [ ]:
# 本模型关节 1 轴竖直，与重力平行，因此 g[0] 恒为 0。
print("g[0] =", sympy.simplify(dyn.g[0]))
print("g 各分量字符数", [len(str(dyn.g[i])) for i in range(robot.dof)])

# 越往末端越短：最后一个关节的重力力矩只能来自第 7 连杆的一阶矩 l_7
sympy.expand(dyn.g[6])


## 4. 导出符号模型

慢速且写文件：下一单元在 `OUT` 中生成 `franka_dynamics.py`、`.c` 和 `.tex`，覆盖同名文件。导出时使用公共子表达式消除（CSE）压缩代码；LaTeX 仅包含 `T/J/g`。


In [ ]:
keys = ("T", "J", "M", "c", "C", "g", "tau", "H", "Hb", "baseparms")

t0 = time.time()
py_path = robot.export.python(OUT / "franka_dynamics.py", keys=keys)
print(f"python {time.time() - t0:6.1f}s", flush=True)

t0 = time.time()
c_path = robot.export.c(OUT / "franka_dynamics.c", keys=keys)
print(f"c      {time.time() - t0:6.1f}s", flush=True)

t0 = time.time()
tex_path = robot.export.latex(OUT / "franka_dynamics.tex", keys=("T", "J", "g"))
print(f"latex  {time.time() - t0:6.1f}s", flush=True)

for p in (py_path, c_path, tex_path):
    print(f"{p.name:24s} {p.stat().st_size / 1024:8.1f} KB")

In [ ]:
# 回读生成的模块，之后的数值全部用它算
import importlib.util

spec = importlib.util.spec_from_file_location("franka_dynamics", py_path)
gen = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gen)

print([n for n in dir(gen) if not n.startswith("_")])
print(f"完整参数 {dyn.n_dynparms} 个 → 基参数 {dyn.n_base} 个")

## 5. 验证生成代码

导入刚生成的 Python 模块，将其力矩、惯性矩阵和回归矩阵结果与 MuJoCo 比较，并检查 `Ṁ - 2C` 的斜对称性质。


In [ ]:
p_true = ref.dynparms()          # MJCF 主轴惯性 → barycentric 参数
pi_b_true = gen.baseparms(p_true)  # 投到基参数空间，作为辨识的比对真值
ddq_test = np.array([0.2, 0.3, -0.5, 0.4, -0.2, 0.1, -0.3])

tau_mj = ref.inverse_dynamics(q_test, dq_test, ddq_test)
tau_gen = gen.tau(q_test, dq_test, ddq_test, p_true)
split = (
    gen.M(q_test, p_true) @ ddq_test
    + gen.c(q_test, dq_test, p_true)
    + gen.g(q_test, p_true)
)

d = mujoco.MjData(model)
d.qpos[:] = q_test
mujoco.mj_forward(model, d)
M_mj = np.zeros((robot.dof, robot.dof))
mujoco.mj_fullM(model, M_mj, d.qM)

print(f"tau        vs mj_inverse  {np.abs(tau_gen - tau_mj).max():.2e}")
print(f"M ddq+c+g  vs mj_inverse  {np.abs(split - tau_mj).max():.2e}")
print(f"M          vs mj_fullM    {np.abs(gen.M(q_test, p_true) - M_mj).max():.2e}")
print(f"H pi       vs mj_inverse  {np.abs(gen.H(q_test, dq_test, ddq_test) @ p_true - tau_mj).max():.2e}")
print(f"Hb pi_b    vs mj_inverse  {np.abs(gen.Hb(q_test, dq_test, ddq_test) @ pi_b_true - tau_mj).max():.2e}")

In [ ]:
# 结构性质：Ṁ - 2C 必须斜对称。3R 上能 sympy.simplify(S + S.T) 直接证，
# 7 轴的 S 有上千万节点，改成随机点上数值验证——Ṁ 用 M 的中心差分
def Mdot(q, dq, parms, eps=1e-5):
    out = np.zeros((robot.dof, robot.dof))
    for k in range(robot.dof):
        qp, qm = q.copy(), q.copy()
        qp[k] += eps
        qm[k] -= eps
        out += (gen.M(qp, parms) - gen.M(qm, parms)) / (2 * eps) * dq[k]
    return out


rng_chk = np.random.default_rng(1)
worst_skew = worst_c = 0.0
for _ in range(5):
    qa = rng_chk.uniform(-1.2, 1.2, robot.dof)
    dqa = rng_chk.uniform(-1.0, 1.0, robot.dof)
    C_num = gen.C(qa, dqa, p_true)
    S = Mdot(qa, dqa, p_true) - 2 * C_num
    worst_skew = max(worst_skew, np.abs(S + S.T).max())
    worst_c = max(worst_c, np.abs(C_num @ dqa - gen.c(qa, dqa, p_true)).max())

print(f"max |S + S^T|   {worst_skew:.2e}   (中心差分本身的误差量级)")
print(f"max |C dq - c|  {worst_c:.2e}   (纯符号，应到机器精度)")

## 6. 解析激励筛选

有限项傅里叶轨迹为

$$q_i(t)=q_{0,i}+\sum_k\left(\frac{a_{ik}}{\omega k}\sin(\omega kt)-\frac{b_{ik}}{\omega k}\cos(\omega kt)\right).$$

`q/dq/ddq` 均由解析式求值。将随机候选缩放进关节限位，堆叠 `Hb` 得到 `W`，列出条件数、秩和最小奇异值，并按条件数排序。训练轨迹的评分仅使用回归矩阵；另一随机种子生成验证轨迹。

下一单元的验证搜索包含 SLSQP 精炼，可能较慢。本页不做闭环试跑；跟踪、饱和和接触筛选由 `robot-identify --robot franka` 执行。


In [ ]:
from robot_model.data import MotionData, add_noise, sample_trajectory
from robot_model.excitation import (
    generate_excitation_candidates,
    regressor_quality,
    search_excitation,
)
from robot_model.identification import (
    DynamicsDecomposer,
    RecoveryConfig,
    identify,
    pack_inertial_parameters,
    parameter_error,
    predict_torque,
    recover_inertial_parameters,
)

lo = model.jnt_range[: robot.dof, 0].copy()
hi = model.jnt_range[: robot.dof, 1].copy()
for i in range(robot.dof):
    print(f"joint{i + 1}: [{lo[i]:7.4f}, {hi[i]:7.4f}]")

Hb_func = gen.Hb  # 直接用导出的代码，不再 lambdify
rng = np.random.default_rng(0)

# 列出候选质量，按条件数排序。
candidates = generate_excitation_candidates(
    robot.dof,
    rng,
    trials=12,
    frequencies=(0.10, 0.15, 0.20),
    harmonics=(3, 4, 5),
    vel_scale=0.8,
    limits=(lo, hi),
)

rows = []
for k, cand in enumerate(candidates):
    t = cand.timestamps(rate=100.0)
    q, dq, ddq = cand.evaluate(t)
    preview = MotionData(t, q, dq, ddq, np.zeros_like(q))
    qlt = regressor_quality(Hb_func, preview)
    rows.append((k, cand.base_freq, cand.n_harmonics, qlt))

rows.sort(key=lambda r: r[3]["condition_number"])
print(f"{'#':>3} {'f0':>5} {'Nh':>3} {'rank':>6} {'cond':>10} {'σ_min/√N':>10}")
for k, f0, nh, qlt in rows[:8]:
    print(
        f"{k:3d} {f0:5.2f} {nh:3d} "
        f"{qlt['rank']:3d}/{qlt['n_parms']:<2d} "
        f"{qlt['condition_number']:10.3g} "
        f"{qlt['min_singular_value']:10.3g}"
    )

best_idx, _, _, best_qlt = rows[0]
train_traj = candidates[best_idx]
train_cond = best_qlt["condition_number"]
print(
    f"\nTRAIN candidate #{best_idx}: cond(W) = {train_cond:.3g}, "
    f"rank = {best_qlt['rank']}/{best_qlt['n_parms']}"
)

# 使用独立随机种子搜索验证轨迹。
val_traj, val_cond = search_excitation(
    Hb_func, robot.dof, np.random.default_rng(1),
    trials=15, vel_scale=0.8, limits=(lo, hi),
)
print(f"VAL   search_excitation: cond(W) = {val_cond:.3g}")

# 后续绘图使用的训练轨迹别名。
traj, cond = train_traj, train_cond


In [ ]:
t_traj = train_traj.timestamps(rate=100.0)
q_traj, dq_traj, ddq_traj = train_traj.evaluate(t_traj)
print("TRAIN in limits:", bool((q_traj >= lo).all() and (q_traj <= hi).all()))
print("joint travel :", np.round(q_traj.max(axis=0) - q_traj.min(axis=0), 3))
print("limit width  :", np.round(hi - lo, 3))
print("peak |dq|    :", np.round(np.abs(dq_traj).max(axis=0), 3))
print("peak |ddq|   :", np.round(np.abs(ddq_traj).max(axis=0), 3))
print(f"train_cond={train_cond:.3g}  val_cond={val_cond:.3g}")


In [ ]:
# 选中的训练激励：画 q / dq / ddq，虚线是关节限位
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for i in range(robot.dof):
    axes[0].plot(t_traj, q_traj[:, i], label=f"q{i+1}")
    axes[1].plot(t_traj, dq_traj[:, i], label=f"dq{i+1}")
    axes[2].plot(t_traj, ddq_traj[:, i], label=f"ddq{i+1}")
for i in range(robot.dof):
    axes[0].axhline(lo[i], color=f"C{i}", ls=":", lw=0.8, alpha=0.7)
    axes[0].axhline(hi[i], color=f"C{i}", ls=":", lw=0.8, alpha=0.7)
axes[0].set_ylabel("q [rad]")
axes[1].set_ylabel("dq [rad/s]")
axes[2].set_ylabel(r"ddq [rad/s$^2$]")
axes[2].set_xlabel("t [s]")
axes[0].set_title(
    f"TRAIN excitation  f0={train_traj.base_freq:.2f} Hz  "
    f"Nh={train_traj.n_harmonics}  cond~{train_cond:.3g}"
)
axes[0].legend(ncol=7, fontsize=8, loc="upper right")
fig.tight_layout()
plt.show()


## 7. 采样、辨识与独立验证

`DATA_MODE` 选择理想数据或带白噪声的合成测量。依次查看训练/验证数据、仅用训练数据辨识，再比较两条轨迹的力矩预测。

本节后面的“保存数据”单元会写入 `OUT / "identification"`，包括傅里叶系数、处理后的训练/验证数据和 JSON 报告；同名文件会被覆盖。


In [ ]:
# 1. 采数：训练轨迹和验证轨迹各采一份
DATA_MODE = "noisy"  # "ideal" | "noisy"

from robot_model.data import filter_measurements

def acquire(trajectory, seed):
    raw = sample_trajectory(trajectory, ref.inverse_dynamics, rate=100)
    if DATA_MODE == "ideal":
        return raw
    if DATA_MODE != "noisy":
        raise ValueError('DATA_MODE must be "ideal" or "noisy"')
    noisy = add_noise(
        raw,
        np.random.default_rng(seed),
        q_std=2e-4,
        tau_std=0.05,
    )
    return filter_measurements(noisy, cutoff=5.0)

train_data = acquire(train_traj, seed=1)
val_data = acquire(val_traj, seed=2)
print(f"DATA_MODE={DATA_MODE}")
print(f"train samples={len(train_data)}  t=[{train_data.t[0]:.2f}, {train_data.t[-1]:.2f}] s")
print(f"val   samples={len(val_data)}  t=[{val_data.t[0]:.2f}, {val_data.t[-1]:.2f}] s")


In [ ]:
# 训练数据预览：此处显示 acquire 返回的处理后数据。
def plot_motion(md, title):
    fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
    series = (
        (md.q, "q [rad]", "q"),
        (md.dq, "dq [rad/s]", "dq"),
        (md.ddq, r"ddq [rad/s$^2$]", "ddq"),
        (md.tau, "tau [N.m]", "tau"),
    )
    for ax, (Y, ylab, prefix) in zip(axes, series):
        for j in range(robot.dof):
            ax.plot(md.t, Y[:, j], label=f"{prefix}{j+1}", lw=1.0)
        ax.set_ylabel(ylab)
    axes[0].set_title(title)
    axes[0].legend(ncol=7, fontsize=8, loc="upper right")
    axes[-1].set_xlabel("t [s]")
    fig.tight_layout()
    plt.show()

plot_motion(
    train_data,
    f"TRAIN acquired data  DATA_MODE={DATA_MODE}  N={len(train_data)}",
)


In [ ]:
# 验证数据预览；这份数据不参与拟合。
plot_motion(
    val_data,
    f"VAL acquired data  DATA_MODE={DATA_MODE}  N={len(val_data)}  cond~{val_cond:.3g}",
)


In [ ]:
# 3. 只用训练数据辨识基参数
result = identify(Hb_func, train_data)
print(result.summary())
print(
    "pi_b relative error (vs MJCF truth):",
    parameter_error(result.parms, pi_b_true)["rel_norm"],
)

def torque_rms(md, parms):
    err = md.tau - predict_torque(Hb_func, md, parms)
    per = np.sqrt(np.mean(err ** 2, axis=0))
    return per, float(np.sqrt(np.mean(err ** 2)))

train_rms, train_rms_all = torque_rms(train_data, result.parms)
val_rms, val_rms_all = torque_rms(val_data, result.parms)
print("train torque RMS [N.m]:", np.round(train_rms, 6), f" overall {train_rms_all:.3e}")
print("val   torque RMS [N.m]:", np.round(val_rms, 6), f" overall {val_rms_all:.3e}")

# 后面分解 / 恢复用的别名
data = train_data
traj = train_traj


In [ ]:
# 4a. 训练力矩：实测 vs 预测
def plot_torque_fit(md, parms, title, overall_ref=None):
    tau_meas = md.tau
    tau_fit = predict_torque(Hb_func, md, parms)
    err = tau_meas - tau_fit
    rms = np.sqrt(np.mean(err ** 2, axis=0))
    fig, axes = plt.subplots(robot.dof, 1, figsize=(10, 1.1 * robot.dof), sharex=True)
    if robot.dof == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.plot(md.t, tau_meas[:, i], label="measured", lw=1.2)
        ax.plot(md.t, tau_fit[:, i], "--", label="predicted", lw=1.0)
        ax.set_ylabel(f"tau{i+1} [N.m]")
        ax.text(
            0.99, 0.85, f"RMS={rms[i]:.2e}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
        )
    axes[0].set_title(title)
    axes[0].legend(loc="upper left", fontsize=8)
    axes[-1].set_xlabel("t [s]")
    fig.tight_layout()
    plt.show()
    print("per-joint torque RMS [N.m]:", np.round(rms, 6))
    print(f"overall RMS {np.sqrt(np.mean(err ** 2)):.3e} N.m")
    if overall_ref is not None:
        print(f"(reference overall RMS {overall_ref:.3e} N.m)")

plot_torque_fit(
    train_data,
    result.parms,
    f"TRAIN torque fit  DATA_MODE={DATA_MODE}  (same data used for identify)",
)


In [ ]:
# 4b. 验证力矩：实测 vs 预测（留出验证，没参与拟合）
plot_torque_fit(
    val_data,
    result.parms,
    f"验证力矩  DATA_MODE={DATA_MODE}（未参与拟合）",
    overall_ref=train_rms_all,
)


In [ ]:
# 保存数据：覆盖 ID_OUT 中的同名 NPZ 和 JSON 文件。
import json

ID_OUT = OUT / "identification"
ID_OUT.mkdir(parents=True, exist_ok=True)

def _traj_dict(tr):
    return {
        "q0": tr.q0.tolist(),
        "a": tr.a.tolist(),
        "b": tr.b.tolist(),
        "base_freq": float(tr.base_freq),
    }

np.savez_compressed(
    ID_OUT / "trajectories.npz",
    train_q0=train_traj.q0, train_a=train_traj.a, train_b=train_traj.b,
    train_freq=train_traj.base_freq,
    val_q0=val_traj.q0, val_a=val_traj.a, val_b=val_traj.b,
    val_freq=val_traj.base_freq,
)
for name, md in (("train", train_data), ("validation", val_data)):
    np.savez_compressed(
        ID_OUT / f"{name}.npz",
        t=md.t, q=md.q, dq=md.dq, ddq=md.ddq, tau=md.tau,
    )

report = {
    "DATA_MODE": DATA_MODE,
    "train_condition_number": float(train_cond),
    "val_condition_number": float(val_cond),
    "n_base": int(result.n_parms),
    "pi_b_hat": result.parms.tolist(),
    "pi_b_true": pi_b_true.tolist(),
    "pi_b_rel_error": float(parameter_error(result.parms, pi_b_true)["rel_norm"]),
    "train_torque_rms_Nm": train_rms.tolist(),
    "train_torque_rms_overall": train_rms_all,
    "val_torque_rms_Nm": val_rms.tolist(),
    "val_torque_rms_overall": val_rms_all,
    "train_trajectory": _traj_dict(train_traj),
    "val_trajectory": _traj_dict(val_traj),
}
(ID_OUT / "report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved to", ID_OUT.resolve())
for f in sorted(ID_OUT.iterdir()):
    print(f"  {f.name:20s} {f.stat().st_size/1024:7.1f} KB")


In [ ]:
# 附带：训练力矩加白噪声，看辨识灵敏度
for std in (0.0, 0.01, 0.05, 0.2):
    noisy = train_data if std == 0 else add_noise(
        train_data, np.random.default_rng(1), tau_std=std
    )
    res = identify(Hb_func, noisy)
    err = parameter_error(res.parms, pi_b_true)["rel_norm"]
    print(
        f"tau_std={std:<5} param rel-err {err:.3e}  "
        f"residual RMS {res.residual_rms:.3e}"
    )


## 8. 从基参数分离 M / c / g

43 个基参数是 70 个刚体参数的线性组合，逐连杆质量、质心和惯量通常不能唯一恢复。给定 `Hb` 和辨识得到的 `π_b`，仍可计算其对应的动力学项：

$$g(q)=H_b(q,0,0)\pi_b,\quad M(q)_{:,i}=H_b(q,0,e_i)\pi_b-g(q),\quad c(q,\dot q)=H_b(q,\dot q,0)\pi_b-g(q).$$

下面在单点与 20 个随机构型上比较分解结果、生成代码和 MuJoCo。默认加噪数据会使辨识模型与真值存在差异。


In [ ]:
dec = DynamicsDecomposer(Hb_func, result.parms, robot.dof)
terms = dec.at(q_test, dq_test)

print(f"M 非对称度 {terms.symmetry_error():.2e}（理论为 0）")
print(f"M   vs 符号     {np.abs(terms.M - gen.M(q_test, p_true)).max():.2e}")
print(f"c   vs 符号     {np.abs(terms.c - gen.c(q_test, dq_test, p_true)).max():.2e}")
print(f"g   vs 符号     {np.abs(terms.g - gen.g(q_test, p_true)).max():.2e}")
print(f"M   vs mj_fullM {np.abs(terms.M - M_mj).max():.2e}")
print(f"tau 重组 vs 真值 {np.abs(terms.torque(ddq_test) - tau_mj).max():.2e}")

In [ ]:
# 在 20 个随机构型上检查分解误差。
rng_sw = np.random.default_rng(7)
errs = {"M": [], "c": [], "g": [], "tau": []}
for _ in range(20):
    qa = rng_sw.uniform(lo, hi)
    dqa = rng_sw.uniform(-1.0, 1.0, robot.dof)
    ddqa = rng_sw.uniform(-2.0, 2.0, robot.dof)
    tm = dec.at(qa, dqa)
    errs["M"].append(np.abs(tm.M - gen.M(qa, p_true)).max())
    errs["c"].append(np.abs(tm.c - gen.c(qa, dqa, p_true)).max())
    errs["g"].append(np.abs(tm.g - gen.g(qa, p_true)).max())
    errs["tau"].append(
        np.abs(tm.torque(ddqa) - ref.inverse_dynamics(qa, dqa, ddqa)).max()
    )

for k, v in errs.items():
    print(f"{k:4s} 最大误差 {max(v):.3e}")


## 9. 可选：恢复逐连杆惯性参数

`RUN_RECOVERY = False` 时跳过求解与后两项评分。开启后，在基参数匹配和物理约束下搜索一组完整参数，保留短预算 `runs=1`、`annealing_maxiter=5`、`local_maxiter=200`；该步骤仍可能较慢，也可能未通过验收。

名义参数从独立加载的原始 MJCF 读取，约束来自 `configs/franka/recovery.json`。配置中的 `nominal_prior_weight` 对名义参数提供弱正则。本页采数与名义参数源自同一个未扰动模型，只演示恢复接口；README 的闭环实验另有对象惯性扰动。评分真值仅在求解后读取，不用于构造初值或约束。

恢复通常不唯一；查看 `success`、物理可行性、基参数匹配和力矩差异。算法来源见 [FrankaEmikaPandaDynModel](https://github.com/JiangShangJiu/FrankaEmikaPandaDynModel)。


In [ ]:
from dataclasses import replace
import json

RUN_RECOVERY = False

nominal_model = mujoco.MjModel.from_xml_path(str(panda_xml_path()))
nominal_ref = MujocoReference(nominal_model, PANDA_MDH_BODY_NAMES)
nominal = pack_inertial_parameters(
    robot,
    [link.m for link in nominal_ref.links],
    [link.r for link in nominal_ref.links],
    [link.I for link in nominal_ref.links],
)
cfg = RecoveryConfig(**json.loads(
    (_ROOT / "configs" / "franka" / "recovery.json").read_text(encoding="utf-8")
))
cfg = replace(cfg, runs=1, annealing_maxiter=5, local_maxiter=200)

if RUN_RECOVERY:
    recovered = recover_inertial_parameters(
        robot, result.parms, nominal_parameters=nominal, config=cfg,
    )
    r = recovered.report
    print(
        f"success={recovered.success}  "
        f"solver={r['solver_success']}  "
        f"feasible={r['physical_feasible']}  "
        f"matches_base={r['matches_base']}  "
        f"base_error={r['base_error']:.3e}"
    )
    if not recovered.success:
        print("恢复未通过验收，后续数值仅供诊断；辨识基参数保留在 result.parms。")
else:
    print("跳过惯性参数恢复；开启 RUN_RECOVERY 后依次运行本节三个单元。")


In [ ]:
if RUN_RECOVERY:
    # 仅在恢复结束后读取评分真值。
    truth_m = np.array([link.m for link in ref.links])
    truth_c = np.array([link.r for link in ref.links])
    truth_I = np.array([link.I for link in ref.links])
    print(f"{'link':>4} {'m_rec':>8} {'m_true':>8} {'|Δm|':>8}  CoM_rec / CoM_true")
    for i in range(robot.dof):
        dm = abs(recovered.masses[i] - truth_m[i])
        print(
            f"{i+1:>4} {recovered.masses[i]:>8.4f} {truth_m[i]:>8.4f} {dm:>8.4f}  "
            f"{np.round(recovered.centers_of_mass[i], 4)} / {np.round(truth_c[i], 4)}"
        )
    print(f"总质量  rec={recovered.masses.sum():.4f}  true={truth_m.sum():.4f}")
    print("\n质心惯性张量对角元 (Ixx, Iyy, Izz)：")
    for i in range(robot.dof):
        print(
            f"  link{i+1}  rec={np.round(np.diag(recovered.inertias_com[i]), 5)}  "
            f"true={np.round(np.diag(truth_I[i]), 5)}"
        )


In [ ]:
if RUN_RECOVERY:
    # 力矩：恢复后的基参数应与辨识基参数几乎等价
    tau_id = predict_torque(Hb_func, data, result.parms)
    tau_rec = predict_torque(Hb_func, data, recovered.base_parameters)
    print(f"辨识 vs 恢复 力矩 RMS {np.sqrt(np.mean((tau_id - tau_rec) ** 2)):.3e}")
    print(f"恢复 vs 测量 力矩 RMS {np.sqrt(np.mean((tau_rec - data.tau) ** 2)):.3e}")

    tau_mj_pt = ref.inverse_dynamics(q_test, dq_test, ddq_test)
    tau_rec_pt = Hb_func(q_test, dq_test, ddq_test) @ recovered.base_parameters
    print(f"单点 |τ_rec - τ_mj| max {np.abs(tau_rec_pt - tau_mj_pt).max():.3e}")


生成代码的回归测试位于 `tests/franka/test_franka_export.py`（`slow`）。论文系数与 SLSQP 激励对照见 [excitation.ipynb](excitation.ipynb)，闭环结果和复现命令见 [项目 README](../../README.md)。
